# Solar Filament Segmentation Challenge 2026 — end-to-end demo

This notebook reproduces our submission from the stored weights (no training). It runs the whole pipeline on the
competition test images, writes `submission.csv`, validates the file, and shows intermediate results for one image.

Pipeline: **detector** (2048 px U-Net ResNet-34, 5 folds × 8-way TTA) → **candidates** (hysteresis + gap bridging) →
**refiner** (UNet++ R34 + ConvNeXt-T, per-candidate crops) → **scorer** (gradient-boosted hurdle model) → 0.5 px erosion → RLE.

Requirements: `pip install -r requirements.txt` (Python 3.12, PyTorch 2.x; a GPU with ≥ 6 GB is recommended, CPU works but takes ~2 min per image).

In [ ]:
import os, sys, time
sys.path.insert(0, os.path.abspath(".."))      # repository root
import numpy as np, torch
from solarfil import config as C
from solarfil.data import list_stems, read_image, write_submission, read_submission
from solarfil.postprocess import overlap_count
sys.path.insert(0, os.path.abspath("../scripts"))
from predict import load_models, predict_image

DATA = "../data/MAGFiLO_1.0_Kaggle_2026"      # <- adjust: extracted Kaggle archive
TEST_IMG = f"{DATA}/test/test_images"
WEIGHTS = "../weights"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| test images:", len(list_stems(TEST_IMG)))

## 1. Load the stored weights

In [ ]:
t0 = time.time()
det, ref, scorer = load_models(WEIGHTS, device)
print(f"{len(det)} detector folds, {len(ref)} refiner models, scorer rule {scorer.rule}  ({time.time() - t0:.1f} s)")

## 2. Run the pipeline on every test image and write the submission

In [ ]:
stems = list_stems(TEST_IMG)
# stems = stems[:5]   # uncomment for a quick check
sub, timing, debug = {}, [], {}
t0 = time.time()
for i, s in enumerate(stems, 1):
    img = read_image(f"{TEST_IMG}/{s}.jpeg")
    final, d = predict_image(img, det, ref, scorer, device, timing)
    sub[s] = final
    if i == 1:
        debug = dict(stem=s, img=img, **d, final=final)
    if i % 20 == 0 or i == len(stems):
        td, tr, ts = np.mean(timing, 0)
        print(f"{i}/{len(stems)}  {time.time() - t0:.0f} s elapsed | per image: detector {td:.2f} s, candidates+refiner {tr:.2f} s, scorer+postproc {ts:.2f} s")
write_submission("submission.csv", sub)
print("filaments:", sum(len(v) for v in sub.values()), "| images with at least one filament:", sum(bool(v) for v in sub.values()))

## 3. Validate the file (format, no overlapping masks, 2048×2048 RLE)

In [ ]:
chk = read_submission("submission.csv")
assert set(chk) == {s for s in sub if sub[s]}
bad = [s for s, rles in chk.items() if overlap_count(rles) > 0]
print("rows:", sum(len(v) for v in chk.values()), "| images with overlapping masks:", len(bad))
assert not bad

## 4. What the stages produce for one image

In [ ]:
import cv2, matplotlib.pyplot as plt
from pycocotools import mask as mu

def label_map(rles):
    lab = np.zeros((C.FULL, C.FULL), np.int32)
    for i, r in enumerate(rles, 1):
        lab[mu.decode(r) > 0] = i
    return lab

def overlay(img, lab):
    rng = np.random.RandomState(3); pal = rng.randint(60, 255, (lab.max() + 2, 3)).astype(np.uint8); pal[0] = 0
    c = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB).astype(np.float32); m = lab > 0
    c[m] = c[m] * 0.3 + pal[lab][m] * 0.7
    return c.astype(np.uint8)

d = debug
panels = [(d["img"], "gray", "H-alpha image"),
          (cv2.resize(d["prob"].astype(np.float32), (C.FULL, C.FULL)), "magma", "Stage 1 probability"),
          (overlay(d["img"], label_map(d["cands"])), None, f"candidates ({len(d['cands'])})"),
          (overlay(d["img"], label_map(d["refined"])), None, f"refined + scored ({len(d['refined'])})"),
          (overlay(d["img"], label_map(d["final"])), None, f"final, eroded 0.5 px ({len(d['final'])})")]
fig, axes = plt.subplots(1, 5, figsize=(25, 5.5))
for ax, (p, cm, t) in zip(axes, panels):
    ax.imshow(p, cmap=cm); ax.set_title(f"{d['stem']}: {t}"); ax.set_axis_off()
plt.tight_layout(); plt.show()

## 5. (Optional) score on training images

The organizers' Panoptic Quality can be computed on training images with `scripts/evaluate.py`. Note that predictions on
training images made with these weights are **in-sample** for 4 of the 5 folds, so the number is optimistic; the
out-of-fold numbers in the report (PQ 0.461) were obtained by training 5 folds and predicting each image with the fold
that did not see it (`scripts/train_detector.py`, `scripts/train_refiner.py`, `scripts/fit_scorer.py`).

In [ ]:
# Example (slow: full pipeline on 707 images):
# !python ../scripts/predict.py --images {DATA}/train/train_images --weights ../weights --out train_pred.csv
# !python ../scripts/evaluate.py --csv train_pred.csv --data {DATA} --per_expert